# 02: Latent Representation Geometry and t-SNE Embedding Analysis

**Project:** Self-Supervised Representation Learning for Histopathological Image Classification Under Limited Label Availability  
**Evaluation Cohort:** CRC-VAL-HE-7K (External Test Set, $N = 7,180$)  
**Objective:** Evaluate and compare the semantic clustering, manifold structure, and class separation of feature embeddings ($h \in \mathbb{R}^{512}$) extracted by:
1. **Supervised ResNet-18 (Random Scratch)**
2. **ImageNet-Pretrained ResNet-18 (Transfer Learning)**
3. **SimCLR Self-Supervised Pretrained ResNet-18 (Linear Probe / Frozen Features)**
4. **SimCLR Self-Supervised Pretrained ResNet-18 (Full Fine-Tuned)**

---

## 1. Setup and Module Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

# Ensure project root in python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils import load_config, get_device, set_seed
from src.datasets import get_downstream_loaders
from src.models import SupervisedResNet18, ResNet18Encoder, LinearProbeClassifier, FineTuneClassifier
from src.evaluate import extract_embeddings
from src.visualization import plot_tsne_embeddings

set_seed(42)
config = load_config(os.path.join(project_root, 'configs', 'config.yaml'))
device = get_device('auto')
class_names = [config['dataset']['classes'][i] for i in range(config['dataset']['num_classes'])]
print(f"Active Device: {device}")
print(f"Target Classes ({len(class_names)}): {class_names}")

## 2. Load External Test Cohort (CRC-VAL-HE-7K)

Strict zero-leakage protocol: Representations are evaluated exclusively on the held-out external test cohort.

In [ ]:
_, _, test_loader, _ = get_downstream_loaders(config, label_fraction=1.0, seed=42)
print(f"Official Test Set Size: {len(test_loader.dataset)} images across {len(class_names)} tissue categories.")

## 3. Extract Penultimate Representations ($h \in \mathbb{R}^{512}$)

We extract 512-dimensional feature embeddings from the penultimate pooling layer across all benchmarked model variants.

In [ ]:
checkpoints_dir = os.path.join(project_root, 'checkpoints')
max_samples = 2000
embeddings_dict = {}

# 1. SimCLR Pretrained Standalone Encoder (Unsupervised SSL)
simclr_ckpt = os.path.join(checkpoints_dir, 'simclr_encoder_best.pth')
if os.path.exists(simclr_ckpt):
    simclr_enc = ResNet18Encoder(pretrained=False).to(device)
    simclr_enc.load_state_dict(torch.load(simclr_ckpt, map_location=device, weights_only=True))
    simclr_enc.eval()
    emb_simclr, y_test = extract_embeddings(simclr_enc, test_loader, device=device, max_samples=max_samples)
    embeddings_dict['SimCLR SSL (Pretrained)'] = emb_simclr
    print("SimCLR SSL embeddings extracted:", emb_simclr.shape)

# 2. ImageNet-Pretrained ResNet-18 (Transfer Baseline)
imagenet_enc = ResNet18Encoder(pretrained=True).to(device)
imagenet_enc.eval()
emb_imagenet, _ = extract_embeddings(imagenet_enc, test_loader, device=device, max_samples=max_samples)
embeddings_dict['ImageNet-1K Transfer'] = emb_imagenet
print("ImageNet embeddings extracted:", emb_imagenet.shape)

# 3. Random Scratch Initialization (Baseline)
scratch_enc = ResNet18Encoder(pretrained=False).to(device)
scratch_enc.eval()
emb_scratch, _ = extract_embeddings(scratch_enc, test_loader, device=device, max_samples=max_samples)
embeddings_dict['Random Initialization'] = emb_scratch
print("Random scratch embeddings extracted:", emb_scratch.shape)

## 4. 2D t-SNE Manifold Projection and Comparative Visualization

In [ ]:
fig, axes = plt.subplots(1, len(embeddings_dict), figsize=(6 * len(embeddings_dict), 5.5))
if len(embeddings_dict) == 1:
    axes = [axes]

palette = sns.color_palette("tab10", n_colors=9)

for idx, (m_name, emb) in enumerate(embeddings_dict.items()):
    ax = axes[idx]
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
    z_2d = tsne.fit_transform(emb)
    
    for c_idx, c_name in enumerate(class_names):
        mask = y_test == c_idx
        ax.scatter(
            z_2d[mask, 0],
            z_2d[mask, 1],
            color=palette[c_idx],
            label=c_name if idx == len(embeddings_dict) - 1 else None,
            alpha=0.6,
            s=16,
            edgecolors="none",
        )
    ax.set_title(f"Latent Space: {m_name}", fontsize=11, fontweight="bold")
    ax.set_xlabel("t-SNE Dimension 1", fontsize=10)
    if idx == 0:
        ax.set_ylabel("t-SNE Dimension 2", fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.3)

fig.legend(
    title="Colorectal Histopathology Tissue Class",
    loc="center left",
    bbox_to_anchor=(0.91, 0.5),
    fontsize=9,
    title_fontsize=10,
)
plt.suptitle(
    "Comparative Representation Geometry on Untouched CRC-VAL-HE-7K Test Cohort",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout(rect=[0, 0, 0.9, 1.0])
plt.show()

## 5. Quantitative Cluster Separation (Silhouette Score Analysis)

In [ ]:
from sklearn.metrics import silhouette_score

print("=" * 60)
print("Quantitative Representation Cluster Separation (Silhouette Score)")
print("=" * 60)
for m_name, emb in embeddings_dict.items():
    score = silhouette_score(emb, y_test, metric='cosine')
    print(f"{m_name:35s} | Cosine Silhouette Score: {score:.4f}")